# Lost in Transcription — Spanish–Nahuatl

Dedicated Spanish–Nahuatl RunPod workflow. Run top-to-bottom: install the compatible GPU stack, download the public training corpus, select by speaker-disjoint competition-dev WER, final-fit, package, then publish the final weights and `submission.zip`. The final competition archive never needs network access.

## 0. RunPod setup (run first on a GPU pod)

This installs the complete Qwen3-ASR, LoRA, packaging, and Hugging Face environment on a fresh Python 3.12 CUDA pod. It is development-only and is not included in the submission archive.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

def find_project_root() -> Path:
    configured = os.environ.get('LIT_ROOT')
    candidates = ([Path(configured)] if configured else []) + [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if all((candidate / item).exists() for item in ('requirements-runpod.txt', 'src', 'spanish_nahuatl')):
            return candidate.resolve()
    raise FileNotFoundError('Could not locate the project root; start Jupyter inside the repository or set LIT_ROOT.')

ROOT = find_project_root()

# Installs requirements-runpod.txt, repairing any apt-managed package without a pip RECORD file
# (blinker, pyparsing, ...), then runs scripts/verify_runpod.py.
subprocess.run([sys.executable, str(ROOT / 'scripts' / 'install_runpod.py')], check=True)
print('RunPod environment verified. Restart the kernel if pip upgraded an imported package.')


## 1. Configure paths and credentials

Set both data directories. `MDC_API_KEY` is used solely to download Tetelancingo; neither it nor the Hugging Face token is stored in any artifact.

In [ ]:
import json
from getpass import getpass

if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

SPNH_DATA = ROOT / 'data' / 'spanish_nahuatl'
SPNH_DEV_DIR = SPNH_DATA / 'development'  # metadata.tsv + clips/
TETEL_DIR = SPNH_DATA / 'tetelancingo'
ARTIFACTS = ROOT / 'artifacts' / 'spnh'
ARTIFACTS.mkdir(parents=True, exist_ok=True)
TENSORBOARD_ROOT = ARTIFACTS / 'tensorboard'
TENSORBOARD_ROOT.mkdir(parents=True, exist_ok=True)

# Preferred: set MDC_API_KEY as a RunPod secret.
if not os.environ.get('MDC_API_KEY'):
    os.environ['MDC_API_KEY'] = getpass('MDC API key (not saved): ')
if not os.environ['MDC_API_KEY'].strip():
    raise RuntimeError('MDC_API_KEY is required to download Tetelancingo')


## 2. Download Tetelancingo

Skip only when `TETEL_DIR` already contains the extracted corpus. The API key must never be copied into the final ZIP.

In [ ]:
if not (TETEL_DIR.exists() and any(p.is_file() and not p.name.startswith('.') for p in TETEL_DIR.rglob('*'))):
    subprocess.run([
        sys.executable, str(ROOT / 'spanish_nahuatl' / 'download_tetelancingo.py'), '--download',
    ], check=True, env=os.environ.copy())
else:
    print(f'Reusing {TETEL_DIR}')


## 3. Build manifests and a held-out speaker split

Use speaker-disjoint competition-dev WER for every selection decision. Tetelancingo’s normalized Nahuatl `sentence` is supervision; its Spanish translation is not.

In [ ]:
from spanish_nahuatl import prepare
from lit_jv import prompts
from lit_jv.decode import DecodeConfig, FallbackConfig, Qwen3AsrDecoder, build_backend
from lit_jv.scoring import wer_report
from lit_jv.train_lora import AugmentSpec, LoraSpec, StageSpec, TrainSpec, merge_and_save, train

dev_rows = prepare.load_competition_dev(SPNH_DEV_DIR)
dev_train, dev_eval = prepare.holdout_speakers(dev_rows, count=1, seed=2026)
tetel_train = prepare.load_tetelancingo(TETEL_DIR, split='train')
tetel_eval = prepare.load_tetelancingo(TETEL_DIR, split='test')
fit_rows = tetel_train + dev_train
terms = prepare.glossary(fit_rows)
(ARTIFACTS / 'glossary.json').write_text(json.dumps(terms, ensure_ascii=False, indent=2), encoding='utf-8')
{'dev_train': len(dev_train), 'dev_eval': len(dev_eval), 'tetel_train': len(tetel_train), 'tetel_eval': len(tetel_eval), 'glossary': len(terms), 'orthography_flags': prepare.orthography_audit(tetel_train)}


## 4. Zero-shot gate

Record this WER. Reject a fine-tune that does not beat it.

In [ ]:
def score_rows(model_path, rows, prompt, name, chunk_size=16):
    backend = build_backend(model_path, prefer='vllm', log=print)
    decoder = Qwen3AsrDecoder(backend, fallback=FallbackConfig(enabled=False), log=print)
    try:
        hypotheses = decoder.decode_pass(rows, DecodeConfig(prompt=prompt, name=name, temperature=0.0), chunk_size=chunk_size)
        report = wer_report([row['raw_text'] for row in rows], [hyp.text for hyp in hypotheses])
        print(report.as_row(name))
        return report
    finally:
        decoder.close()

baseline_prompt = prompts.PromptConfig(prose='codeswitch_spnh', style_tag=prompts.STYLE_VOICE_NOTE_SP_NH, glossary=tuple(terms), max_glossary_terms=128, force_language='Spanish')
baseline = score_rows('Qwen/Qwen3-ASR-1.7B', dev_eval, baseline_prompt, 'qwen-zero-shot')


## 5. Train the selected recipe

Stage 1 learns Nahuatl vocabulary/acoustics from Tetelancingo; stage 2 prioritizes Spanish–Nahuatl voice-note style. Generation WER selects checkpoints; teacher-forced evaluation remains disabled.

In [ ]:
spec = TrainSpec(
    model_path='Qwen/Qwen3-ASR-1.7B', output_dir=str(ARTIFACTS / 'lora'), target_language='Spanish',
    lora=LoraSpec(r=16, alpha=32, adapt_text_decoder=True, adapt_audio_encoder=False),
    augment=AugmentSpec(corpora=('tetelancingo',)), prompt=baseline_prompt,
    eval_steps=0, wer_eval_steps=10, wer_eval_clips=len(dev_eval), wer_eval_batch_size=4, run_name='spnh-qwen3asr',
)
stages = (
    StageSpec('s1-tetelancingo', {'tetelancingo': 1.0, 'spnh_dev': 1.25}, epochs=2.0, learning_rate=1.0e-4, batch_size=4, grad_accumulation=8),
    StageSpec('s2-voice-note', {'spnh_dev': 1.0, 'tetelancingo': 0.15}, epochs=5.0, learning_rate=3.0e-5, batch_size=4, grad_accumulation=8),
)
selected = train(fit_rows, spec, stages=stages, save_adapter_as='selected', eval_rows=dev_eval)
selected['best_wer']


## 6. Select, final-fit, and package

If the assertion fails, stop and change one variable before rerunning from the baseline. Once it passes, final fitting freezes the recipe and uses every released training row.

In [ ]:
merged = merge_and_save(spec, ARTIFACTS / 'lora' / 'selected', ARTIFACTS / 'merged')
selected_dev = score_rows(merged, dev_eval, spec.prompt, 'selected-dev')
selected_tetel = score_rows(merged, tetel_eval, spec.prompt, 'selected-tetel')
assert selected_dev.wer < baseline.wer, 'Reject this recipe: it did not beat zero-shot Qwen.'

final_rows = prepare.load_tetelancingo(TETEL_DIR, split=None) + dev_rows
final_spec = TrainSpec(**{**spec.__dict__, 'output_dir': str(ARTIFACTS / 'final-lora'), 'wer_eval_steps': 0})
train(final_rows, final_spec, stages=stages, save_adapter_as='final', eval_rows=None)
final_merged = merge_and_save(final_spec, ARTIFACTS / 'final-lora' / 'final', ARTIFACTS / 'final-merged')
package = subprocess.run([
    sys.executable, str(ROOT / 'spanish_nahuatl' / 'package_submission.py'),
    '--model', str(final_merged), '--glossary', str(ARTIFACTS / 'glossary.json'),
], check=True, capture_output=True, text=True)
print(package.stdout)
SUBMISSION_ZIP = ARTIFACTS / 'submission.zip'
assert SUBMISSION_ZIP.is_file() and SUBMISSION_ZIP.stat().st_size > 0
print(f'Ready to publish: {final_merged} and {SUBMISSION_ZIP}')


## 7. Publish final model weights and `submission.zip` to Hugging Face

Set `SPNH_HF_REPO` to an `owner/repository` destination if desired. Otherwise, this cell creates `lost-in-transcription-spanish-nahuatl` under the authenticated account. `HF_TOKEN` is requested privately if it is not already a RunPod secret; it is never saved to a model, ZIP, or notebook output.

In [ ]:
from huggingface_hub import HfApi, hf_hub_url

if not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = getpass('Hugging Face write token (not saved): ')
if not os.environ['HF_TOKEN'].strip():
    raise RuntimeError('HF_TOKEN is required to publish the model and submission archive')

api = HfApi(token=os.environ['HF_TOKEN'])
account = api.whoami()['name']
REPO_ID = os.environ.get('SPNH_HF_REPO', f'{account}/lost-in-transcription-spanish-nahuatl')
PRIVATE = os.environ.get('SPNH_HF_PRIVATE', '0') == '1'
api.create_repo(repo_id=REPO_ID, repo_type='model', private=PRIVATE, exist_ok=True)

# Publish final-merged exactly once under model/, and the competition archive at the root.
model_commit = api.upload_folder(
    repo_id=REPO_ID, repo_type='model', folder_path=str(final_merged), path_in_repo='model',
    commit_message='Upload final Spanish-Nahuatl merged model weights',
    ignore_patterns=['**/__pycache__/**', '*.lock'],
)
archive_commit = api.upload_file(
    path_or_fileobj=str(SUBMISSION_ZIP), path_in_repo='submission.zip', repo_id=REPO_ID, repo_type='model',
    commit_message='Upload Spanish-Nahuatl competition submission archive',
)
api.upload_file(
    path_or_fileobj=str(ARTIFACTS / 'glossary.json'), path_in_repo='assets/glossary.json', repo_id=REPO_ID, repo_type='model',
    commit_message='Upload final glossary provenance',
)
print('Model weights:', getattr(model_commit, 'commit_url', model_commit))
print('Submission archive:', getattr(archive_commit, 'commit_url', archive_commit))
print('Repository:', f'https://huggingface.co/{REPO_ID}')
print('Archive download:', hf_hub_url(repo_id=REPO_ID, filename='submission.zip', repo_type='model'))
